In [2]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Load annotations
df = pd.read_csv("human_validation_oracle.csv")

# ------------------------------------------------------------------
# Create AST similarity buckets
# ------------------------------------------------------------------
def similarity_bucket(score):
    if score == 1.0:
        return "Type-1 (Identical, S=1.0)"
    elif 0.9 <= score < 1.0:
        return "Type-2 (Renamed, 0.9<=S<1.0)"
    elif 0.7 <= score < 0.9:
        return "Type-3 (Modified, 0.7<=S<0.9)"
    else:
        return "Structural Drift/Stale Metadata (S<0.7)"

df["bucket"] = df["live_ast_similarity_score"].apply(similarity_bucket)

# ------------------------------------------------------------------
# Calculate inter-annotator agreement
# ------------------------------------------------------------------
df["agree"] = (
    df["label_1"].astype(str).str.strip()
    ==
    df["label_2"].astype(str).str.strip()
)

# ------------------------------------------------------------------
# Agreement statistics per bucket
# ------------------------------------------------------------------
summary = (
    df.groupby("bucket")
      .agg(
          Total=("agree", "size"),
          Agreements=("agree", "sum")
      )
      .reset_index()
)

summary["Human Agreement (%)"] = (
    summary["Agreements"] / summary["Total"] * 100
).round(1)

# ------------------------------------------------------------------
# Order rows like the paper
# ------------------------------------------------------------------
bucket_order = [
    "Type-1 (Identical, S=1.0)",
    "Type-2 (Renamed, 0.9<=S<1.0)",
    "Type-3 (Modified, 0.7<=S<0.9)",
    "Structural Drift/Stale Metadata (S<0.7)"
]

summary["bucket"] = pd.Categorical(
    summary["bucket"],
    categories=bucket_order,
    ordered=True
)

summary = summary.sort_values("bucket")

# ------------------------------------------------------------------
# Print results
# ------------------------------------------------------------------
print("\nHuman Agreement by AST Similarity Bucket\n")
print(summary[["bucket", "Total", "Agreements", "Human Agreement (%)"]])

# ------------------------------------------------------------------
# Optional: generate LaTeX-ready rows
# ------------------------------------------------------------------
print("\nLaTeX rows:\n")

for _, row in summary.iterrows():
    print(
        f"{row['bucket']} & "
        f"{row['Total']:,} & "
        f"{row['Human Agreement (%)']:.1f}\\% \\\\"
    )


Human Agreement by AST Similarity Bucket

                                    bucket  Total  Agreements  \
1                Type-1 (Identical, S=1.0)    400         392   
2             Type-2 (Renamed, 0.9<=S<1.0)    250         224   
3            Type-3 (Modified, 0.7<=S<0.9)    200         150   
0  Structural Drift/Stale Metadata (S<0.7)    150         113   

   Human Agreement (%)  
1                 98.0  
2                 89.6  
3                 75.0  
0                 75.3  

LaTeX rows:

Type-1 (Identical, S=1.0) & 400 & 98.0\% \\
Type-2 (Renamed, 0.9<=S<1.0) & 250 & 89.6\% \\
Type-3 (Modified, 0.7<=S<0.9) & 200 & 75.0\% \\
Structural Drift/Stale Metadata (S<0.7) & 150 & 75.3\% \\


In [3]:
results = []

for bucket in bucket_order:
    subset = df[df["bucket"] == bucket]

    agreement = (
        (subset["label_1"].astype(str).str.strip() ==
         subset["label_2"].astype(str).str.strip())
        .mean() * 100
    )

    kappa = cohen_kappa_score(
        subset["label_1"].astype(str).str.strip(),
        subset["label_2"].astype(str).str.strip()
    )

    results.append([
        bucket,
        len(subset),
        round(agreement, 1),
        round(kappa, 4)
    ])

kappa_df = pd.DataFrame(
    results,
    columns=[
        "Classification",
        "N",
        "Agreement (%)",
        "Cohen's Kappa"
    ]
)

print(kappa_df)

                            Classification    N  Agreement (%)  Cohen's Kappa
0                Type-1 (Identical, S=1.0)  400           98.0         0.2650
1             Type-2 (Renamed, 0.9<=S<1.0)  250           89.6         0.3970
2            Type-3 (Modified, 0.7<=S<0.9)  200           75.0         0.0980
3  Structural Drift/Stale Metadata (S<0.7)  150           75.3         0.4705


In [4]:
print("=== Loading Replication Artifacts ===")
prov_df = pd.read_csv("downsized_clone_provenance_dataset.csv")
sim_df = pd.read_csv("clone_pair_similarity.csv")
oracle_df = pd.read_csv("human_validation_oracle.csv")

# 1. Total Count Verification
print(f"Loaded Provenance Trace Records: {len(prov_df):,}")
print(f"Loaded Deep AST Metric Rows: {len(sim_df):,}")
print(f"Loaded Human Oracle Assessment Subset: {len(oracle_df):,}")
print("-" * 40)

# 2. Inter-Rater Evaluation Metrics
# Calculating consensus directly from explicit rater verification labels
matching_annotations = (oracle_df["label_1"] == oracle_df["label_2"]).sum()
agreement_pct = (matching_annotations / len(oracle_df)) * 100
print(f"Expert Consensus Convergence Baseline: {agreement_pct:.2f}%")

# 3. Structural Variance Breakdown (Type-2 Mutation Isolation)
type_2_renames = prov_df[prov_df["source_method_name"] != prov_df["sink_method_name"]]
print(f"Identified Structural Type-2 (Renamed) Links: {len(type_2_renames):,}")
print("=======================================")

=== Loading Replication Artifacts ===
Loaded Provenance Trace Records: 100,000
Loaded Deep AST Metric Rows: 25,753
Loaded Human Oracle Assessment Subset: 1,000
----------------------------------------
Expert Consensus Convergence Baseline: 87.90%
Identified Structural Type-2 (Renamed) Links: 0
